# Fazendo análise para entender a lógica necessária

In [ ]:
# Importando bibliotecas:
from datetime import datetime
import pandas as pd
import unicodedata
import numpy as np
import re

In [ ]:
# Busca pela pasta com as bases:
pasta_movimentacoes = R"..\..\..\Base de Dados Original\parquet\Movimentacoes\Movimentacoes_01-01-2023_09-30-2025.parquet"

# Carregando as bases:
df_movimentacoes = pd.read_parquet(pasta_movimentacoes)

In [ ]:
# Mostrando o DataFrame:
df_movimentacoes.head(0)

In [ ]:
# Corrigindo a ordem das colunas:
df_movimentacoes = df_movimentacoes[
    ['Nº Pessoal', 'Abreviação UO antiga', 'Desc. UO antiga', 'Desc. Cargo antigo',
    'Abreviação UO', 'Desc. UO', 'Desc. Cargo', 'Tipo da Movimentação', 'Data de alteração']
]

# Mostrando o DataFrame:
df_movimentacoes.head(0)

In [ ]:
# Garante que a coluna está em formato datetime
df_movimentacoes["Data de alteração"] = pd.to_datetime(df_movimentacoes["Data de alteração"], errors="coerce", dayfirst=True)

# Filtra apenas os registros com data >= 2023
df_teste_antes = df_movimentacoes[df_movimentacoes["Data de alteração"] < datetime(2023, 1, 1)]
df_teste_depois = df_movimentacoes[df_movimentacoes["Data de alteração"] > datetime(2025, 9, 30)]

# Exemplo de resultado
print(f"Total filtrado: {len(df_teste_antes)}")
print(f"Total filtrado: {len(df_teste_depois)}")

In [ ]:
# Corrigindo relacionamento entre colunas:
df_movimentacoes = df_movimentacoes[
    (df_movimentacoes['Abreviação UO antiga'] != df_movimentacoes['Abreviação UO'])
]

df = df_movimentacoes[
    (df_movimentacoes['Abreviação UO antiga'].str.startswith('B-RL', na=False)) &
   ~(df_movimentacoes['Abreviação UO'].str.startswith('B-RL', na=False))
]

df

In [ ]:
def tabela_saidas_areas(df_movimentacoes: pd.DataFrame, agrupar_por_prefixo: bool = True,
                        anos=(2023, 2024, 2025)) -> pd.DataFrame:
    df = df_movimentacoes.copy()

    # 1) Apenas quem trocou de área (mesma UO = só cargo, ignora)
    df = df.loc[
        df['Abreviação UO antiga'].notna() &
        df['Abreviação UO'].notna() &
        (df['Abreviação UO antiga'] != df['Abreviação UO'])
    ].copy()

    # 2) Ano
    df['Ano'] = pd.to_datetime(df['Data de alteração'], errors='coerce', dayfirst=True).dt.year

    # 3) Área de saída
    if agrupar_por_prefixo:
        def _prefixo(s):
            s = str(s)
            m = re.match(r'^([A-Za-z]-[A-Za-z]+)', s)
            return m.group(1) if m else s
        df['Area'] = df['Abreviação UO antiga'].map(_prefixo)
    else:
        df['Area'] = df['Abreviação UO antiga']

    # 4) Contagem analítica (pessoas únicas) por área/ano
    base = (
        df.groupby(['Area', 'Ano'])['Nº Pessoal']
          .nunique()
          .rename('Qtd')
          .reset_index()
    )

    # 5) Wide e colunas fixas de anos
    wide = base.pivot(index='Area', columns='Ano', values='Qtd')

    if anos is not None:
        wide = wide.reindex(columns=list(anos), fill_value=0)

    wide = wide.fillna(0).astype(int).reset_index()
    # renomeia anos para string
    wide.columns = ['Area'] + [str(c) for c in wide.columns[1:]]
    return wide

In [ ]:
df_movimentacoes = tabela_saidas_areas(df_movimentacoes, agrupar_por_prefixo=True)  # -> colunas: Area, 2023, 2024, 2025
df_movimentacoes.head()